<div style="background:#1F3864;padding:22px 28px;border-radius:10px;margin-bottom:14px">
<h2 style="color:#A8C8E8;margin:0 0 6px">Digitalization, AI &amp; XAI in Healthcare</h2>
<h1 style="color:#FFFFFF;margin:0 0 10px;font-size:1.45em">NB23 — Clinical XAI Dashboard with Streamlit</h1>
<p style="color:#BDD7EE;margin:0;font-size:.95em"><strong>Module 6: Application Workshop: Design of an Explainable AI Solution</strong></p>
<p style="color:#9DC3E6;margin:0;font-size:.85em">
Datasets: Cleveland Heart Disease (UCI) . Pima Indians Diabetes (UCI) . Custom CSV upload<br>
XAI Methods: SHAP . LIME . Counterfactual -- selectable by the user at runtime
</p>
</div>

## Learning Objectives

1. **Understand** the five core Streamlit patterns: `cache_resource`, `sidebar`, `tabs`, `session_state`, `download_button`
2. **Build** a complete multi-dataset clinical XAI dashboard in a single Python file (`app_nb23.py`)
3. **Apply all 6 XAI methods** from the course -- SHAP, LIME, MAPLE, Counterfactual, Surrogate Tree, and GEMEX -- selectable by the user
4. **Compare methods** side-by-side on the same patient using the new Compare tab
5. **Support CSV upload** so the dashboard works with built-in or custom clinical datasets
6. **Deploy** the app locally, on Google Colab (ngrok), and on Streamlit Community Cloud

> **All 6 XAI methods in one dashboard:**
> SHAP . LIME . MAPLE . Counterfactual . Surrogate Tree . GEMEX

## Setup

```bash
pip install streamlit shap lime scikit-learn pandas numpy matplotlib gemex
```

GEMEX is optional -- the dashboard runs without it (GEMEX tab shows an install message).

**Dataset files** (place in same folder as `app_nb23.py`):
- `cleveland_heart.csv` -- https://raw.githubusercontent.com/jbrownlee/Datasets/master/heart-disease.csv
- `pima_diabetes.csv` -- https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data

Both are built into the app as synthetic fallback if the files are absent.

---
## Section 1 -- Five Core Streamlit Concepts

Before reading `app_nb23.py`, understand these five patterns.
Every one appears with a labelled `# CONCEPT N` comment inside the app.

In [1]:
# Run this cell to review the five Streamlit concepts
print("""
CONCEPT 1 -- @st.cache_resource
Caches heavy objects (models, explainers) across all reruns and users.
Runs ONCE on first load -- no retraining on every widget click.

  @st.cache_resource(show_spinner="Training model...")
  def build_model():
      gbm = GradientBoostingClassifier().fit(X, y)
      return gbm

CONCEPT 2 -- st.sidebar
Persistent left panel. Controls visible on every tab.
Holds: Dataset selector, Patient slider, XAI method radio (6 methods)

CONCEPT 3 -- st.tabs
Four tabs: Patient View | Compare XAI Methods | Dataset Explorer | Model Info

  tab1, tab2, tab3, tab4 = st.tabs(["Patient View", "Compare", "Explorer", "Info"])

CONCEPT 4 -- st.session_state
Persists explanation history across reruns.

  if "history" not in st.session_state:
      st.session_state.history = []

CONCEPT 5 -- st.download_button
One-line export of XAI report (.txt) and plot (.png).
""")
print("Five Streamlit concepts reviewed.")
print()
print("6 XAI methods available in app_nb23.py:")
print("  1. SHAP           -- Shapley values")
print("  2. LIME           -- Local linear surrogate")
print("  3. MAPLE          -- RF co-membership surrogate")
print("  4. Counterfactual -- Nearest opposite patient")
print("  5. Surrogate Tree -- Global IF-THEN rules")
print("  6. GEMEX          -- Geodesic information-geometric")


CONCEPT 1 -- @st.cache_resource
Caches heavy objects (models, explainers) across all reruns and users.
Runs ONCE on first load -- no retraining on every widget click.

  @st.cache_resource(show_spinner="Training model...")
  def build_model():
      gbm = GradientBoostingClassifier().fit(X, y)
      return gbm

CONCEPT 2 -- st.sidebar
Persistent left panel. Controls visible on every tab.
Holds: Dataset selector, Patient slider, XAI method radio (6 methods)

CONCEPT 3 -- st.tabs
Four tabs: Patient View | Compare XAI Methods | Dataset Explorer | Model Info

  tab1, tab2, tab3, tab4 = st.tabs(["Patient View", "Compare", "Explorer", "Info"])

CONCEPT 4 -- st.session_state
Persists explanation history across reruns.

  if "history" not in st.session_state:
      st.session_state.history = []

CONCEPT 5 -- st.download_button
One-line export of XAI report (.txt) and plot (.png).

Five Streamlit concepts reviewed.

6 XAI methods available in app_nb23.py:
  1. SHAP           -- Shapley values


---
## Section 2 -- Backend Verification

Run this cell to verify both dataset backends produce correct AUC scores
before building the full app. The same loader and model code appears inside
`@st.cache_resource` functions in `app_nb23.py`.

In [2]:
import warnings; warnings.filterwarnings('ignore')
import os, numpy as np, pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

COLS_CLEVELAND = ['age','sex','cp','trestbps','chol','fbs','restecg',
                  'thalach','exang','oldpeak','slope','ca','thal','target']
COLS_PIMA      = ['Pregnancies','Glucose','BloodPressure','SkinThickness',
                  'Insulin','BMI','DiabetesPedigreeFunction','Age','Outcome']

def load_cleveland():
    if os.path.exists('cleveland_heart.csv'):
        df = pd.read_csv('cleveland_heart.csv')
        if df.columns[0] != 'age': df.columns = COLS_CLEVELAND
        df = df.replace('?', np.nan).apply(pd.to_numeric, errors='coerce').dropna()
        df['target'] = (df['target'] > 0).astype(int)
        return df, 'target', [c for c in df.columns if c != 'target']
    rng = np.random.default_rng(42); n = 303
    df = pd.DataFrame({'age':rng.integers(30,80,n).astype(float),
        'sex':rng.integers(0,2,n).astype(float),'cp':rng.integers(0,4,n).astype(float),
        'trestbps':rng.normal(130,20,n),'chol':rng.normal(246,50,n),
        'fbs':rng.integers(0,2,n).astype(float),'restecg':rng.integers(0,3,n).astype(float),
        'thalach':rng.normal(150,23,n),'exang':rng.integers(0,2,n).astype(float),
        'oldpeak':np.abs(rng.normal(1.0,1.2,n)),'slope':rng.integers(0,3,n).astype(float),
        'ca':rng.integers(0,4,n).astype(float),'thal':rng.integers(1,4,n).astype(float)})
    df['target']=((df['age']>55).astype(int)+(df['chol']>250).astype(int)+(df['trestbps']>140).astype(int)>1).astype(int)
    print('  Warning: using synthetic Cleveland data')
    return df,'target',[c for c in df.columns if c!='target']

def load_pima():
    if os.path.exists('pima_diabetes.csv'):
        raw = pd.read_csv('pima_diabetes.csv',header=0)
        raw = raw[raw.iloc[:,0].astype(str)!=raw.columns[0]]
        if raw.columns[0]!='Pregnancies': raw.columns=COLS_PIMA
        df = raw.apply(pd.to_numeric,errors='coerce').dropna()
        df['Outcome']=df['Outcome'].astype(int)
        return df,'Outcome',[c for c in df.columns if c!='Outcome']
    rng=np.random.default_rng(42);n=768
    df=pd.DataFrame({'Pregnancies':rng.integers(0,18,n).astype(float),
        'Glucose':rng.normal(120,32,n),'BloodPressure':rng.normal(69,19,n),
        'SkinThickness':rng.normal(20,16,n),'Insulin':np.abs(rng.normal(80,115,n)),
        'BMI':rng.normal(32,7,n),'DiabetesPedigreeFunction':np.abs(rng.normal(0.47,0.33,n)),
        'Age':rng.integers(21,82,n).astype(float)})
    df['Outcome']=((df['Glucose']>140).astype(int)+(df['BMI']>35).astype(int)+(df['Age']>50).astype(int)>1).astype(int)
    print('  Warning: using synthetic Pima data')
    return df,'Outcome',[c for c in df.columns if c!='Outcome']

def train_gbm(df,tgt,feats):
    X=df[feats].values.astype(np.float32); y=df[tgt].values.astype(int)
    sc=StandardScaler(); X_sc=sc.fit_transform(X)
    gbm=GradientBoostingClassifier(n_estimators=150,learning_rate=0.1,
                                    max_depth=3,min_samples_leaf=5,random_state=42)
    cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
    y_cv=cross_val_predict(gbm,X_sc,y,cv=cv,method='predict_proba')[:,1]
    auc=roc_auc_score(y,y_cv); gbm.fit(X_sc,y)
    return gbm,sc,auc

print('Verifying backends...')
df_c,tgt_c,feat_c=load_cleveland(); gbm_c,sc_c,auc_c=train_gbm(df_c,tgt_c,feat_c)
print(f'Cleveland Heart  -- {len(df_c)} patients | AUC: {auc_c:.3f}')
df_p,tgt_p,feat_p=load_pima(); gbm_p,sc_p,auc_p=train_gbm(df_p,tgt_p,feat_p)
print(f'Pima Diabetes    -- {len(df_p)} patients | AUC: {auc_p:.3f}')
print()
print('Checking GEMEX...')
try:
    import gemex; print(f'GEMEX {gemex.__version__} available')
except ImportError:
    print('GEMEX not installed (pip install gemex) -- dashboard runs without it')
print()
print('Both backends verified. Section 3 writes app_nb23.py.')

Verifying backends...
Cleveland Heart  -- 297 patients | AUC: 0.855
Pima Diabetes    -- 768 patients | AUC: 0.823

Checking GEMEX...
GEMEX 1.2.2 available

Both backends verified. Section 3 writes app_nb23.py.


---
## Section 3 -- The Complete Streamlit Application

The cell below writes `app_nb23.py` to disk. Run it then launch with:
```bash
python -m streamlit run app_nb23.py
```

**App layout (4 tabs):**
- **Tab 1 -- Patient View:** Prediction banner + feature table + XAI chart + method description + session history + download
- **Tab 2 -- Compare XAI Methods:** Run all 6 methods on one patient, see agreement table and all plots side-by-side
- **Tab 3 -- Dataset Explorer:** Data table + class distribution + feature statistics
- **Tab 4 -- Model Info:** AUC + GBM feature importance + full XAI method reference guide

**6 XAI methods (sidebar radio):**

| Method | Type | Key property |
|--------|------|--------------|
| SHAP | Feature attribution | Theoretically grounded, deterministic |
| LIME | Local linear surrogate | Fast, model-agnostic, can be unstable |
| MAPLE | RF co-membership surrogate | More stable than LIME |
| Counterfactual | Nearest opposite patient | Most actionable clinically |
| Surrogate Tree | Global IF-THEN rules | Human-readable, printable |
| GEMEX | Geodesic information-geometric | Fisher-Rao deviation from reference |

In [3]:
app_code = "# app_nb23.py -- Clinical XAI Dashboard\n# NB23 . Digitalization, AI & XAI in Healthcare . Module 6\n#\n# Datasets : Cleveland Heart Disease (UCI) . Pima Indians Diabetes (UCI)\n# XAI      : SHAP . LIME . Counterfactual . MAPLE . Surrogate Tree . GEMEX\n# Run      : streamlit run app_nb23.py\n\nimport warnings; warnings.filterwarnings(\"ignore\")\nimport os, io\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use(\"Agg\")\nimport matplotlib.pyplot as plt\nimport streamlit as st\nimport shap\nimport lime\nimport lime.lime_tabular\nfrom sklearn.ensemble import GradientBoostingClassifier, RandomForestRegressor\nfrom sklearn.tree import DecisionTreeClassifier, export_text, plot_tree\nfrom sklearn.model_selection import StratifiedKFold, cross_val_predict\nfrom sklearn.metrics import roc_auc_score\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.linear_model import Ridge\nfrom datetime import datetime\n\n# Try GEMEX\ntry:\n    import gemex\n    GEMEX_OK = True\nexcept ImportError:\n    GEMEX_OK = False\n\n# ---- Page config -------------------------------------------------------------\nst.set_page_config(\n    page_title=\"Clinical XAI Dashboard -- NB23\",\n    page_icon=\"[+]\",\n    layout=\"wide\",\n    initial_sidebar_state=\"expanded\",\n)\n\nNAVY   = \"#1F3864\"; BLUE  = \"#2E75B6\"; GREEN = \"#1F7A5C\"\nRED    = \"#C0392B\"; ORANGE= \"#D4860B\"; PURPLE= \"#7B3F9E\"\nTEAL   = \"#117A8B\"; BROWN = \"#6D4C41\"\n\nCOLS_CLEVELAND = [\"age\",\"sex\",\"cp\",\"trestbps\",\"chol\",\"fbs\",\"restecg\",\n                  \"thalach\",\"exang\",\"oldpeak\",\"slope\",\"ca\",\"thal\",\"target\"]\nCOLS_PIMA      = [\"Pregnancies\",\"Glucose\",\"BloodPressure\",\"SkinThickness\",\n                  \"Insulin\",\"BMI\",\"DiabetesPedigreeFunction\",\"Age\",\"Outcome\"]\n\nFEAT_LABELS_CLEVELAND = {\n    \"age\":\"Age (years)\", \"sex\":\"Sex (1=Male)\", \"cp\":\"Chest Pain Type\",\n    \"trestbps\":\"Resting BP (mmHg)\", \"chol\":\"Cholesterol (mg/dL)\",\n    \"fbs\":\"Fasting Blood Sugar>120\", \"restecg\":\"Rest ECG\",\n    \"thalach\":\"Max Heart Rate\", \"exang\":\"Exercise Angina\",\n    \"oldpeak\":\"ST Depression\", \"slope\":\"ST Slope\",\n    \"ca\":\"Major Vessels Coloured\", \"thal\":\"Thalassemia\"\n}\nFEAT_LABELS_PIMA = {\n    \"Pregnancies\":\"Pregnancies\", \"Glucose\":\"Plasma Glucose\",\n    \"BloodPressure\":\"Blood Pressure (mmHg)\", \"SkinThickness\":\"Skin Thickness (mm)\",\n    \"Insulin\":\"Serum Insulin (uU/mL)\", \"BMI\":\"BMI (kg/m2)\",\n    \"DiabetesPedigreeFunction\":\"Diabetes Pedigree\", \"Age\":\"Age (years)\"\n}\n\n# =============================================================================\n# CONCEPT 1 -- @st.cache_resource\n# Trains models once. All reruns reuse the cached objects.\n# =============================================================================\n@st.cache_resource(show_spinner=\"Training Cleveland Heart model...\")\ndef build_cleveland():\n    if os.path.exists(\"cleveland_heart.csv\"):\n        df = pd.read_csv(\"cleveland_heart.csv\")\n        if df.columns[0] != \"age\": df.columns = COLS_CLEVELAND\n        df = df.replace(\"?\", np.nan).apply(pd.to_numeric, errors=\"coerce\").dropna()\n        df[\"target\"] = (df[\"target\"] > 0).astype(int)\n    else:\n        rng = np.random.default_rng(42); n = 303\n        df = pd.DataFrame({\n            \"age\":rng.integers(30,80,n).astype(float),\n            \"sex\":rng.integers(0,2,n).astype(float),\n            \"cp\":rng.integers(0,4,n).astype(float),\n            \"trestbps\":rng.normal(130,20,n),\n            \"chol\":rng.normal(246,50,n),\n            \"fbs\":rng.integers(0,2,n).astype(float),\n            \"restecg\":rng.integers(0,3,n).astype(float),\n            \"thalach\":rng.normal(150,23,n),\n            \"exang\":rng.integers(0,2,n).astype(float),\n            \"oldpeak\":np.abs(rng.normal(1.0,1.2,n)),\n            \"slope\":rng.integers(0,3,n).astype(float),\n            \"ca\":rng.integers(0,4,n).astype(float),\n            \"thal\":rng.integers(1,4,n).astype(float),\n        })\n        df[\"target\"] = ((df[\"age\"]>55).astype(int)+(df[\"chol\"]>250).astype(int)+(df[\"trestbps\"]>140).astype(int)>1).astype(int)\n    feats = [c for c in df.columns if c != \"target\"]\n    X = df[feats].values.astype(np.float32)\n    y = df[\"target\"].values.astype(int)\n    sc = StandardScaler(); X_sc = sc.fit_transform(X)\n    gbm = GradientBoostingClassifier(n_estimators=150,learning_rate=0.1,\n                                      max_depth=3,min_samples_leaf=5,random_state=42)\n    cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)\n    y_cv = cross_val_predict(gbm, X_sc, y, cv=cv, method=\"predict_proba\")[:,1]\n    auc  = roc_auc_score(y, y_cv)\n    gbm.fit(X_sc, y)\n    return dict(df=df, feats=feats, X=X_sc, y=y, gbm=gbm, sc=sc, auc=auc,\n                label_names=[\"No Disease\",\"Heart Disease\"],\n                feat_labels=FEAT_LABELS_CLEVELAND)\n\n@st.cache_resource(show_spinner=\"Training Pima Diabetes model...\")\ndef build_pima():\n    if os.path.exists(\"pima_diabetes.csv\"):\n        raw = pd.read_csv(\"pima_diabetes.csv\", header=0)\n        raw = raw[raw.iloc[:,0].astype(str) != raw.columns[0]]\n        if raw.columns[0] != \"Pregnancies\": raw.columns = COLS_PIMA\n        df = raw.apply(pd.to_numeric, errors=\"coerce\").dropna()\n        df[\"Outcome\"] = df[\"Outcome\"].astype(int)\n    else:\n        rng = np.random.default_rng(42); n = 768\n        df = pd.DataFrame({\n            \"Pregnancies\":rng.integers(0,18,n).astype(float),\n            \"Glucose\":rng.normal(120,32,n),\n            \"BloodPressure\":rng.normal(69,19,n),\n            \"SkinThickness\":rng.normal(20,16,n),\n            \"Insulin\":np.abs(rng.normal(80,115,n)),\n            \"BMI\":rng.normal(32,7,n),\n            \"DiabetesPedigreeFunction\":np.abs(rng.normal(0.47,0.33,n)),\n            \"Age\":rng.integers(21,82,n).astype(float),\n        })\n        df[\"Outcome\"] = ((df[\"Glucose\"]>140).astype(int)+(df[\"BMI\"]>35).astype(int)+(df[\"Age\"]>50).astype(int)>1).astype(int)\n    feats = [c for c in df.columns if c != \"Outcome\"]\n    X = df[feats].values.astype(np.float32)\n    y = df[\"Outcome\"].values.astype(int)\n    sc = StandardScaler(); X_sc = sc.fit_transform(X)\n    gbm = GradientBoostingClassifier(n_estimators=150,learning_rate=0.1,\n                                      max_depth=3,min_samples_leaf=5,random_state=42)\n    cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)\n    y_cv = cross_val_predict(gbm, X_sc, y, cv=cv, method=\"predict_proba\")[:,1]\n    auc  = roc_auc_score(y, y_cv)\n    gbm.fit(X_sc, y)\n    return dict(df=df, feats=feats, X=X_sc, y=y, gbm=gbm, sc=sc, auc=auc,\n                label_names=[\"No Diabetes\",\"Diabetes\"],\n                feat_labels=FEAT_LABELS_PIMA)\n\ndef build_custom(df_raw):\n    df = df_raw.apply(pd.to_numeric, errors=\"coerce\").dropna()\n    target = df.columns[-1]\n    feats  = list(df.columns[:-1])\n    X = df[feats].values.astype(np.float32)\n    y = (df[target].values > df[target].median()).astype(int)\n    sc = StandardScaler(); X_sc = sc.fit_transform(X)\n    n_splits = min(5, max(2, int(y.sum())))\n    gbm = GradientBoostingClassifier(n_estimators=100,learning_rate=0.1,\n                                      max_depth=3,random_state=42)\n    cv  = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)\n    y_cv = cross_val_predict(gbm, X_sc, y, cv=cv, method=\"predict_proba\")[:,1]\n    auc  = roc_auc_score(y, y_cv)\n    gbm.fit(X_sc, y)\n    return dict(df=df, feats=feats, X=X_sc, y=y, gbm=gbm, sc=sc, auc=auc,\n                label_names=[\"Low\",\"High\"], feat_labels={f:f for f in feats})\n\n# =============================================================================\n# SHARED CHART HELPER\n# =============================================================================\ndef make_bar_chart(labels, values, title, xlabel,\n                   pos_color=GREEN, neg_color=RED, figsize=(7,4)):\n    n = len(labels)\n    fig, ax = plt.subplots(figsize=(figsize[0], max(figsize[1], n*0.38)))\n    fig.patch.set_facecolor(\"#0F1923\"); ax.set_facecolor(\"#0F1923\")\n    colors = [pos_color if v >= 0 else neg_color for v in values]\n    idx    = np.argsort(np.abs(values))\n    ax.barh([labels[i] for i in idx], [values[i] for i in idx],\n             color=[colors[i] for i in idx], edgecolor=\"none\")\n    ax.axvline(0, color=\"white\", lw=0.8)\n    ax.set_xlabel(xlabel, color=\"white\")\n    ax.set_title(title, color=\"white\", fontweight=\"bold\", fontsize=10)\n    ax.tick_params(colors=\"white\")\n    ax.spines[\"bottom\"].set_color(\"#444\"); ax.spines[\"left\"].set_color(\"#444\")\n    for sp in [\"top\",\"right\"]: ax.spines[sp].set_visible(False)\n    plt.tight_layout()\n    return fig\n\n# =============================================================================\n# XAI METHOD 1 -- SHAP (NB2)\n# Tree-based Shapley values -- gold standard for tabular GBM\n# =============================================================================\ndef run_shap(backend, idx):\n    gbm=backend[\"gbm\"]; X=backend[\"X\"]\n    feats=backend[\"feats\"]; flab=backend[\"feat_labels\"]\n    expl = shap.TreeExplainer(gbm)\n    sv   = expl.shap_values(X[idx:idx+1])[0]\n    labels = [flab.get(f,f) for f in feats]\n    fig = make_bar_chart(labels, sv,\n                          \"SHAP -- Feature Attributions\",\n                          \"SHAP value (impact on prediction)\")\n    desc = (\n        \"SHAP (SHapley Additive exPlanations) uses game-theory Shapley values to \"\n        \"fairly distribute the prediction gap among features. \"\n        \"Green bars push the prediction toward the positive class; \"\n        \"red bars push it toward the negative class. \"\n        \"Values sum exactly to the total prediction shift from the base rate.\"\n    )\n    return fig, dict(zip(feats, sv.tolist())), desc\n\n# =============================================================================\n# XAI METHOD 2 -- LIME (NB3)\n# Local Interpretable Model-agnostic Explanations\n# =============================================================================\ndef run_lime(backend, idx):\n    gbm=backend[\"gbm\"]; X=backend[\"X\"]\n    feats=backend[\"feats\"]; flab=backend[\"feat_labels\"]\n    def predict_fn(arr): return gbm.predict_proba(arr)\n    expl = lime.lime_tabular.LimeTabularExplainer(\n        X, feature_names=feats, class_names=backend[\"label_names\"],\n        mode=\"classification\", random_state=42)\n    exp  = expl.explain_instance(X[idx], predict_fn,\n                                  num_features=len(feats), num_samples=500)\n    lime_dict = dict(exp.as_list())\n    sv = np.zeros(len(feats))\n    for i,f in enumerate(feats):\n        for k,v in lime_dict.items():\n            if f in k: sv[i]=v; break\n    labels = [flab.get(f,f) for f in feats]\n    fig = make_bar_chart(labels, sv,\n                          \"LIME -- Local Feature Weights\",\n                          \"LIME weight (local attribution)\")\n    desc = (\n        \"LIME perturbs this patient's features and fits a local linear model nearby. \"\n        \"Weights show which features most influenced the prediction in this \"\n        \"specific neighbourhood. Note: LIME can be unstable -- running it twice \"\n        \"may give slightly different results (unlike SHAP which is deterministic).\"\n    )\n    return fig, dict(zip(feats, sv.tolist())), desc\n\n# =============================================================================\n# XAI METHOD 3 -- MAPLE (NB3)\n# More stable local explanations via Random Forest leaf co-membership\n# =============================================================================\ndef run_maple(backend, idx):\n    gbm=backend[\"gbm\"]; X=backend[\"X\"]\n    feats=backend[\"feats\"]; flab=backend[\"feat_labels\"]\n    y=backend[\"y\"]\n    # MAPLE: fit RF, use leaf co-membership to weight training points,\n    # then fit a local Ridge regression\n    rf = RandomForestRegressor(n_estimators=100, random_state=42)\n    rf.fit(X, y.astype(float))\n    leaves = rf.apply(X)              # (n_patients, n_trees)\n    x_leaves = rf.apply(X[idx:idx+1]) # (1, n_trees)\n    # Weight = fraction of trees where patient shares a leaf with x\n    weights = (leaves == x_leaves).mean(axis=1).astype(np.float64)\n    weights /= (weights.sum() + 1e-10)\n    # Fit weighted Ridge on raw features\n    ridge = Ridge(alpha=1.0)\n    ridge.fit(X, gbm.predict_proba(X)[:,1], sample_weight=weights)\n    coefs = ridge.coef_\n    labels = [flab.get(f,f) for f in feats]\n    fig = make_bar_chart(labels, coefs,\n                          \"MAPLE -- Local Ridge Coefficients\",\n                          \"Coefficient (local importance)\",\n                          pos_color=TEAL, neg_color=BROWN)\n    desc = (\n        \"MAPLE (Model-Agnostic Pseudo-Local Explanations) uses Random Forest \"\n        \"leaf co-membership to identify the most similar training patients, \"\n        \"then fits a local weighted Ridge regression. It is more stable than LIME \"\n        \"because the neighbourhood is defined by the model's own structure, \"\n        \"not random perturbations.\"\n    )\n    return fig, dict(zip(feats, coefs.tolist())), desc\n\n# =============================================================================\n# XAI METHOD 4 -- Surrogate Decision Tree (NB4)\n# Global interpretable surrogate -- human-readable rules\n# =============================================================================\ndef run_surrogate_tree(backend, idx):\n    gbm=backend[\"gbm\"]; X=backend[\"X\"]\n    feats=backend[\"feats\"]; flab=backend[\"feat_labels\"]\n    # Train a shallow decision tree to mimic the GBM globally\n    y_soft = gbm.predict_proba(X)[:,1]\n    y_hard = (y_soft > 0.5).astype(int)\n    tree   = DecisionTreeClassifier(max_depth=4, min_samples_leaf=10, random_state=42)\n    tree.fit(X, y_hard)\n    fidelity = (tree.predict(X) == y_hard).mean()\n    # Visualise the tree\n    fig, ax = plt.subplots(figsize=(14, 5))\n    fig.patch.set_facecolor(\"#0F1923\"); ax.set_facecolor(\"#0F1923\")\n    plot_tree(tree,\n              feature_names=[flab.get(f,f) for f in feats],\n              class_names=backend[\"label_names\"],\n              filled=True, rounded=True, fontsize=7,\n              ax=ax,\n              proportion=False,\n              impurity=False)\n    ax.set_title(\n        f\"Surrogate Decision Tree (depth=4, fidelity={fidelity:.1%})\\n\"\n        f\"Approximates the GBM globally with human-readable IF-THEN rules\",\n        color=\"white\", fontweight=\"bold\", fontsize=10, pad=10)\n    fig.patch.set_facecolor(\"#0F1923\")\n    plt.tight_layout()\n    # Also get the text rule for this patient\n    path  = tree.decision_path(X[idx:idx+1])\n    rules = export_text(tree,\n                        feature_names=[flab.get(f,f) for f in feats])\n    # Find which path the current patient takes\n    node_ids = path.indices\n    all_rules = rules.split(\"\\n\")\n    patient_pred = backend[\"label_names\"][int(tree.predict(X[idx:idx+1])[0])]\n    desc = (\n        f\"A shallow Decision Tree (depth=4) trained to mimic the GBM -- \"\n        f\"fidelity {fidelity:.1%}. \"\n        f\"It produces human-readable IF-THEN rules that approximate the black-box model globally. \"\n        f\"For Patient {idx}, the surrogate predicts: {patient_pred}. \"\n        f\"Clinical use: the tree rules can be printed on a clinical decision aid card.\"\n    )\n    vals = dict(zip(feats, tree.feature_importances_.tolist()))\n    return fig, vals, desc\n\n# =============================================================================\n# XAI METHOD 5 -- Counterfactual (NB4)\n# Nearest opposite patient -- actionable clinical advice\n# =============================================================================\ndef run_counterfactual(backend, idx):\n    gbm=backend[\"gbm\"]; X=backend[\"X\"]\n    y=backend[\"y\"]; feats=backend[\"feats\"]\n    flab=backend[\"feat_labels\"]\n    x0=X[idx]; pred0=gbm.predict([x0])[0]\n    opposite = np.where(y != pred0)[0]\n    if len(opposite) == 0: return None, {}, \"No counterfactual found.\"\n    cf_idx = opposite[np.argmin(np.linalg.norm(X[opposite]-x0, axis=1))]\n    delta  = X[cf_idx] - x0\n    labels = [flab.get(f,f) for f in feats]\n    top8   = np.argsort(np.abs(delta))[-8:]\n    fig    = make_bar_chart(\n        [labels[i] for i in top8], [delta[i] for i in top8],\n        f\"Counterfactual -- What would flip the prediction?\\n\"\n        f\"({backend['label_names'][pred0]} -> {backend['label_names'][1-pred0]})\",\n        \"Feature change (CF minus original, standardised)\",\n        pos_color=ORANGE, neg_color=PURPLE)\n    desc = (\n        \"The Counterfactual is the nearest patient in the dataset who received \"\n        \"the opposite prediction. The bars show which features differ and by how much. \"\n        \"Clinically: this answers 'what would need to change for this patient \"\n        \"to be classified differently?' -- the most actionable form of explanation.\"\n    )\n    return fig, dict(zip(feats, delta.tolist())), desc\n\n# =============================================================================\n# XAI METHOD 6 -- GEMEX (NB22)\n# Geodesic Entropic Manifold Explainability\n# Information-geometric arc-length from the reference manifold\n# =============================================================================\ndef run_gemex(backend, idx):\n    gbm=backend[\"gbm\"]; X=backend[\"X\"]\n    feats=backend[\"feats\"]; flab=backend[\"feat_labels\"]\n    y=backend[\"y\"]\n\n    if not GEMEX_OK:\n        return None, {}, \"GEMEX not installed. Run: pip install gemex\"\n\n    # Build numpy head from GBM\n    # GEMEX needs a predict function on embeddings\n    # We use GBM leaf node counts as embeddings\n    X_leaves_3d = gbm.apply(X)  # GradientBoosting.apply returns (n, n_estimators, 1)\n    X_leaves = X_leaves_3d[:, :, 0].astype(np.float32)  # reshape to 2D (n, n_estimators)\n    from sklearn.preprocessing import normalize\n    X_emb = normalize(X_leaves, norm=\"l2\")\n\n    # Reference: negative class patients (label=0)\n    ref_idx = np.where(y == 0)[0][:20]\n    X_ref   = X_emb[ref_idx]\n\n    # Simple predict function on embeddings via Ridge\n    ridge_emb = Ridge(alpha=1.0)\n    ridge_emb.fit(X_emb, gbm.predict_proba(X)[:,1])\n    def predict_fn(arr):\n        arr = np.atleast_2d(arr)\n        scores = ridge_emb.predict(arr)\n        scores = np.clip(scores, 0, 1)\n        return np.column_stack([1-scores, scores])\n\n    try:\n        gx = gemex.Explainer(\n            predict_fn,\n            data_type=\"tabular\",\n            feature_names=[f\"leaf_{i}\" for i in range(X_emb.shape[1])],\n            class_names=backend[\"label_names\"],\n            config=gemex.GemexConfig(\n                n_geodesic_steps=6,\n                n_reference_samples=min(10, len(X_ref)),\n                interaction_order=1,\n                verbose=False))\n        r   = gx.explain(X_emb[idx], X_reference=X_ref)\n        arc = float(r.geodesic_lengths[-1])\n        gsf = np.abs(r.gsf_scores)\n\n        # Map GSF scores back to original features via correlation\n        feat_importance = np.zeros(len(feats))\n        for fi in range(len(feats)):\n            col = X[:, fi]\n            # Correlate each feature with GBM leaf activations\n            leaf_corrs = np.abs([np.corrcoef(col, X_leaves[:,li])[0,1]\n                                  for li in range(X_leaves.shape[1])])\n            # Weight by GSF\n            n_gsf = min(len(gsf), len(leaf_corrs))\n            feat_importance[fi] = np.dot(leaf_corrs[:n_gsf], gsf[:n_gsf] / (gsf[:n_gsf].sum()+1e-10))\n\n        feat_importance = feat_importance / (feat_importance.max()+1e-10)\n\n        labels = [flab.get(f,f) for f in feats]\n        fig = make_bar_chart(labels, feat_importance,\n                              f\"GEMEX -- Geodesic Feature Sensitivity\\nArc-length from reference manifold: {arc:.4f}\",\n                              \"Geodesic sensitivity (0=low, 1=high)\",\n                              pos_color=PURPLE, neg_color=TEAL)\n        desc = (\n            f\"GEMEX (Geodesic Entropic Manifold Explainability) measures how far \"\n            f\"this patient's model embedding lies from the reference manifold \"\n            f\"(negative class patients) in Fisher-Rao information space. \"\n            f\"Arc-length = {arc:.4f} -- \"\n            f\"{'high: patient is structurally distant from the reference group' if arc > 0.3 else 'low: patient is close to the reference group'}. \"\n            f\"Bars show which original features contribute most to this geodesic deviation.\"\n        )\n        return fig, dict(zip(feats, feat_importance.tolist())), desc\n\n    except Exception as e:\n        return None, {}, f\"GEMEX error: {str(e)}\"\n\ndef fig_to_bytes(fig):\n    buf = io.BytesIO()\n    fig.savefig(buf, format=\"png\", dpi=150, bbox_inches=\"tight\",\n                facecolor=fig.get_facecolor())\n    buf.seek(0); return buf.read()\n\n# =============================================================================\n# METHOD REGISTRY -- add new methods here to extend the dashboard\n# =============================================================================\nXAI_METHODS = {\n    \"SHAP\":             (run_shap,           GREEN,  \"NB2\", \"Feature attribution (Shapley values)\"),\n    \"LIME\":             (run_lime,           BLUE,   \"NB3\", \"Local linear surrogate\"),\n    \"MAPLE\":            (run_maple,          TEAL,   \"NB3\", \"Stable local surrogate (RF co-membership)\"),\n    \"Counterfactual\":   (run_counterfactual, ORANGE, \"NB4\", \"Nearest opposite patient\"),\n    \"Surrogate Tree\":   (run_surrogate_tree, BROWN,  \"NB4\", \"Global IF-THEN rules (depth-4 tree)\"),\n    \"GEMEX\":            (run_gemex,          PURPLE, \"NB22\",\"Geodesic information-geometric deviation\"),\n}\n\n# =============================================================================\n# CONCEPT 2 -- st.sidebar\n# =============================================================================\nwith st.sidebar:\n    st.markdown(\n        f'<div style=\"background:{NAVY};padding:14px;border-radius:8px;margin-bottom:12px\">'\n        f'<h3 style=\"color:#E8E8F0;margin:0;font-size:1em\">Clinical XAI Dashboard</h3>'\n        f'<p style=\"color:#9999BB;margin:4px 0 0;font-size:.78em\">NB23 . Module 6 . 6 XAI Methods</p>'\n        f'</div>', unsafe_allow_html=True)\n\n    dataset_choice = st.selectbox(\n        \"Dataset\",\n        [\"Cleveland Heart Disease\", \"Pima Indians Diabetes\", \"Upload CSV\"],\n        help=\"Choose a built-in dataset or upload your own CSV\")\n\n    uploaded = None\n    if dataset_choice == \"Upload CSV\":\n        uploaded = st.file_uploader(\n            \"Upload CSV (last column = target)\", type=[\"csv\"],\n            help=\"Numeric CSV. Last column treated as target variable.\")\n\n    st.markdown(\"---\")\n    xai_method = st.radio(\n        \"XAI Method\",\n        list(XAI_METHODS.keys()),\n        help=\"\\n\".join([f\"{k}: {v[3]}\" for k,v in XAI_METHODS.items()]))\n\n    method_color = XAI_METHODS[xai_method][1]\n    method_nb    = XAI_METHODS[xai_method][2]\n    method_desc  = XAI_METHODS[xai_method][3]\n\n    st.markdown(\n        f'<div style=\"background:#131326;padding:8px;border-radius:6px;'\n        f'border-left:3px solid {method_color};margin-top:6px\">'\n        f'<span style=\"color:{method_color};font-size:.80em;font-weight:bold\">{xai_method}</span>'\n        f'<br><span style=\"color:#9999BB;font-size:.74em\">{method_desc}</span>'\n        f'<br><span style=\"color:#555;font-size:.70em\">Introduced in {method_nb}</span>'\n        f'</div>', unsafe_allow_html=True)\n\n    st.markdown(\"---\")\n    st.markdown(\n        f'<div style=\"background:#131326;padding:8px;border-radius:6px;font-size:.74em;color:#9999BB\">'\n        f'<b style=\"color:#E8E8F0\">All 6 XAI Methods</b><br>'\n        + \"\".join([\n            f'<span style=\"color:{v[1]}\">. {k}</span> ({v[3][:25]}...)<br>'\n            for k,v in XAI_METHODS.items()])\n        + f'</div>', unsafe_allow_html=True)\n\n# ---- Load backend ------------------------------------------------------------\nif dataset_choice == \"Cleveland Heart Disease\":\n    backend = build_cleveland(); ds_name = \"Cleveland Heart Disease\"; ds_color = RED\nelif dataset_choice == \"Pima Indians Diabetes\":\n    backend = build_pima(); ds_name = \"Pima Indians Diabetes\"; ds_color = BLUE\nelse:\n    if uploaded is None:\n        st.info(\"Upload a CSV file in the sidebar to continue.\"); st.stop()\n    df_raw  = pd.read_csv(uploaded)\n    backend = build_custom(df_raw)\n    ds_name = uploaded.name; ds_color = ORANGE\n\nn_patients = len(backend[\"df\"])\n\nwith st.sidebar:\n    patient_idx = st.slider(\"Patient index\", 0, n_patients-1, 0,\n                             help=\"Select a patient from the dataset\")\n    st.caption(f\"Dataset: {n_patients} patients . AUC: {backend['auc']:.3f}\")\n\n# ---- Header ------------------------------------------------------------------\nst.markdown(\n    f'<div style=\"background:{NAVY};padding:16px 20px;border-radius:8px;margin-bottom:16px\">'\n    f'<h2 style=\"color:#FFFFFF;margin:0;font-size:1.3em\">Clinical XAI Dashboard</h2>'\n    f'<p style=\"color:#A8C8E8;margin:4px 0 0;font-size:.88em\">'\n    f'Dataset: <b>{ds_name}</b> . '\n    f'Model AUC: <b>{backend[\"auc\"]:.3f}</b> . '\n    f'XAI Method: <b style=\"color:{method_color}\">{xai_method}</b>'\n    f'</p></div>', unsafe_allow_html=True)\n\n# =============================================================================\n# CONCEPT 3 -- st.tabs\n# =============================================================================\ntab1, tab2, tab3, tab4 = st.tabs([\n    \"Patient View\",\n    \"Compare XAI Methods\",\n    \"Dataset Explorer\",\n    \"Model Info\"\n])\n\n# ---- TAB 1: Patient View -----------------------------------------------------\nwith tab1:\n    row        = backend[\"df\"].iloc[patient_idx]\n    x_raw      = backend[\"X\"][patient_idx]\n    prob       = backend[\"gbm\"].predict_proba([x_raw])[0]\n    pred_class = int(np.argmax(prob))\n    confidence = float(prob[pred_class])\n    true_label = int(backend[\"y\"][patient_idx])\n    label_names= backend[\"label_names\"]\n    pred_color = RED if pred_class == 1 else GREEN\n    correct    = \"Correct\" if pred_class == true_label else \"Incorrect\"\n\n    st.markdown(\n        f'<div style=\"background:{pred_color}22;border:2px solid {pred_color};'\n        f'border-radius:8px;padding:14px 18px;margin-bottom:16px\">'\n        f'<span style=\"color:{pred_color};font-size:1.2em;font-weight:bold\">'\n        f'Prediction: {label_names[pred_class]}</span>'\n        f'&nbsp;&nbsp;&nbsp;'\n        f'<span style=\"color:#E8E8F0\">Confidence: {confidence:.1%}</span>'\n        f'&nbsp;&nbsp;&nbsp;'\n        f'<span style=\"color:#9999BB\">True label: {label_names[true_label]} . {correct}</span>'\n        f'</div>', unsafe_allow_html=True)\n\n    col_feat, col_xai = st.columns([1, 1.6])\n\n    with col_feat:\n        st.subheader(f\"Patient {patient_idx} -- Features\")\n        flab = backend[\"feat_labels\"]\n        feat_df = pd.DataFrame({\n            \"Feature\": [flab.get(f,f) for f in backend[\"feats\"]],\n            \"Value\":   [f\"{row[f]:.2f}\" for f in backend[\"feats\"]]\n        })\n        st.dataframe(feat_df, use_container_width=True, hide_index=True)\n\n    with col_xai:\n        st.subheader(f\"{xai_method} Explanation\")\n        with st.spinner(f\"Computing {xai_method}...\"):\n            run_fn = XAI_METHODS[xai_method][0]\n            fig, vals, explanation_desc = run_fn(backend, patient_idx)\n\n        if fig is not None:\n            st.pyplot(fig, use_container_width=True); plt.close(fig)\n        else:\n            st.warning(explanation_desc)\n\n        if explanation_desc:\n            st.markdown(\n                f'<div style=\"background:#131326;padding:10px;border-radius:6px;'\n                f'border-left:3px solid {method_color};margin-top:8px\">'\n                f'<p style=\"color:#9999BB;font-size:.80em;margin:0\">'\n                f'<b style=\"color:{method_color}\">How to read this:</b> {explanation_desc}'\n                f'</p></div>', unsafe_allow_html=True)\n\n    # CONCEPT 4 -- st.session_state\n    if \"history\" not in st.session_state:\n        st.session_state.history = []\n    if fig is not None:\n        entry = {\"time\": datetime.now().strftime(\"%H:%M:%S\"),\n                 \"patient\": patient_idx, \"method\": xai_method,\n                 \"prediction\": label_names[pred_class],\n                 \"confidence\": f\"{confidence:.1%}\", \"dataset\": ds_name}\n        if not st.session_state.history or \\\n           (st.session_state.history[-1][\"patient\"] != patient_idx or\n            st.session_state.history[-1][\"method\"] != xai_method):\n            st.session_state.history.append(entry)\n            if len(st.session_state.history) > 5:\n                st.session_state.history.pop(0)\n\n    if st.session_state.history:\n        st.markdown(\"---\")\n        st.subheader(\"Session History (last 5 explanations)\")\n        st.dataframe(pd.DataFrame(st.session_state.history[::-1]),\n                     use_container_width=True, hide_index=True)\n\n    # CONCEPT 5 -- st.download_button\n    st.markdown(\"---\")\n    if fig is not None and vals:\n        flab = backend[\"feat_labels\"]\n        report = [\n            \"Clinical XAI Report -- NB23\",\n            f\"Generated : {datetime.now().strftime('%Y-%m-%d %H:%M')}\",\n            f\"Dataset   : {ds_name}\",\n            f\"Patient   : {patient_idx}\",\n            f\"Method    : {xai_method}\",\n            f\"Prediction: {label_names[pred_class]} ({confidence:.1%})\",\n            f\"True label: {label_names[true_label]}\",\n            f\"Model AUC : {backend['auc']:.3f}\",\n            \"\",\n            f\"Method description: {explanation_desc}\",\n            \"\",\n            \"Feature scores (sorted by magnitude):\",\n        ]\n        for f,v in sorted(vals.items(), key=lambda x: abs(x[1]), reverse=True):\n            report.append(f\"  {flab.get(f,f):35s}: {v:+.4f}\")\n        report_text = \"\\n\".join(report)\n\n        col_dl1, col_dl2 = st.columns(2)\n        with col_dl1:\n            st.download_button(\n                label=\"Download XAI Report (.txt)\", data=report_text,\n                file_name=f\"xai_p{patient_idx}_{xai_method.lower().replace(' ','_')}.txt\",\n                mime=\"text/plain\")\n        with col_dl2:\n            st.download_button(\n                label=\"Download Plot (.png)\", data=fig_to_bytes(fig),\n                file_name=f\"xai_p{patient_idx}_{xai_method.lower().replace(' ','_')}.png\",\n                mime=\"image/png\")\n\n# ---- TAB 2: Compare XAI Methods (NEW) ----------------------------------------\nwith tab2:\n    st.subheader(f\"Compare All 6 XAI Methods -- Patient {patient_idx}\")\n    st.markdown(\n        f'<p style=\"color:#9999BB;font-size:.85em\">'\n        f'Run all six methods on Patient {patient_idx} and see how they agree or disagree. '\n        f'Agreement across methods increases clinical confidence; disagreement signals '\n        f'areas where the model behaviour is complex or ambiguous.</p>',\n        unsafe_allow_html=True)\n\n    run_comparison = st.button(\"Run All 6 Methods on this Patient\", type=\"primary\")\n\n    if run_comparison:\n        results = {}\n        prog = st.progress(0)\n        status = st.empty()\n        for i, (mname, (mfn, mcol, mnb, mdesc)) in enumerate(XAI_METHODS.items()):\n            status.text(f\"Running {mname}...\")\n            try:\n                mfig, mvals, mdescription = mfn(backend, patient_idx)\n                results[mname] = (mfig, mvals, mcol, mdescription)\n            except Exception as e:\n                results[mname] = (None, {}, mcol, f\"Error: {str(e)}\")\n            prog.progress((i+1)/len(XAI_METHODS))\n        status.text(\"All methods complete.\")\n\n        # Show top feature per method in a comparison table\n        feats  = backend[\"feats\"]\n        flab   = backend[\"feat_labels\"]\n        comp_rows = []\n        for mname, (mfig, mvals, mcol, mdescription) in results.items():\n            if mvals:\n                top_f = max(mvals, key=lambda k: abs(mvals[k]))\n                top_v = mvals[top_f]\n                comp_rows.append({\n                    \"Method\":       mname,\n                    \"Notebook\":     XAI_METHODS[mname][2],\n                    \"Top Feature\":  flab.get(top_f, top_f),\n                    \"Score\":        f\"{top_v:+.4f}\",\n                    \"Direction\":    \"Risk+\" if top_v > 0 else \"Risk-\",\n                })\n        if comp_rows:\n            st.markdown(\"**Top feature identified by each method:**\")\n            comp_df = pd.DataFrame(comp_rows)\n            st.dataframe(comp_df, use_container_width=True, hide_index=True)\n            # Check agreement\n            top_feats = [r[\"Top Feature\"] for r in comp_rows]\n            majority  = max(set(top_feats), key=top_feats.count)\n            agree_n   = top_feats.count(majority)\n            st.info(\n                f\"Majority agreement: **{majority}** identified as top feature \"\n                f\"by {agree_n}/{len(top_feats)} methods. \"\n                f\"{'Strong agreement -- high confidence in this feature.' if agree_n >= 4 else 'Mixed agreement -- model behaviour is complex for this patient.'}\"\n            )\n\n        st.markdown(\"---\")\n        st.markdown(\"**Explanation plots for all methods:**\")\n        cols = st.columns(2)\n        for i, (mname, (mfig, mvals, mcol, mdescription)) in enumerate(results.items()):\n            with cols[i % 2]:\n                st.markdown(\n                    f'<div style=\"border:1px solid {mcol};border-radius:6px;'\n                    f'padding:8px;margin-bottom:8px\">'\n                    f'<b style=\"color:{mcol}\">{mname}</b>'\n                    f'<span style=\"color:#555;font-size:.72em\"> . {XAI_METHODS[mname][2]}</span>'\n                    f'</div>', unsafe_allow_html=True)\n                if mfig is not None:\n                    st.pyplot(mfig, use_container_width=True); plt.close(mfig)\n                else:\n                    st.warning(mdescription)\n\n# ---- TAB 3: Dataset Explorer -------------------------------------------------\nwith tab3:\n    df   = backend[\"df\"]\n    feats= backend[\"feats\"]\n    tgt  = [c for c in df.columns if c not in feats][0]\n    flab = backend[\"feat_labels\"]\n\n    st.subheader(f\"Dataset: {ds_name}\")\n    c1,c2,c3 = st.columns(3)\n    c1.metric(\"Patients\", len(df))\n    c2.metric(\"Features\", len(feats))\n    c3.metric(\"Positive class rate\", f\"{df[tgt].mean():.1%}\")\n\n    st.dataframe(df.head(50), use_container_width=True)\n    st.markdown(\"---\")\n\n    col_dist, col_stats = st.columns(2)\n    with col_dist:\n        st.subheader(\"Class Distribution\")\n        counts = df[tgt].value_counts().sort_index()\n        fig_d, ax_d = plt.subplots(figsize=(4,3))\n        fig_d.patch.set_facecolor(\"#0F1923\"); ax_d.set_facecolor(\"#0F1923\")\n        bars = ax_d.bar([backend[\"label_names\"][int(i)] for i in counts.index],\n                         counts.values, color=[GREEN, RED][:len(counts)])\n        ax_d.bar_label(bars, color=\"white\")\n        ax_d.tick_params(colors=\"white\")\n        ax_d.set_title(\"Class Distribution\", color=\"white\", fontweight=\"bold\")\n        for sp in [\"top\",\"right\"]: ax_d.spines[sp].set_visible(False)\n        ax_d.spines[\"bottom\"].set_color(\"#444\"); ax_d.spines[\"left\"].set_color(\"#444\")\n        st.pyplot(fig_d, use_container_width=True); plt.close(fig_d)\n    with col_stats:\n        st.subheader(\"Feature Statistics\")\n        st.dataframe(df[feats].describe().T.round(2), use_container_width=True)\n\n# ---- TAB 4: Model Info -------------------------------------------------------\nwith tab4:\n    st.subheader(\"Model Performance\")\n    c1,c2,c3 = st.columns(3)\n    c1.metric(\"Model\", \"Gradient Boosting\")\n    c2.metric(\"CV AUC\", f\"{backend['auc']:.3f}\")\n    c3.metric(\"Patients\", len(backend[\"df\"]))\n\n    st.markdown(\"---\")\n    st.subheader(\"GBM Feature Importance\")\n    fi   = backend[\"gbm\"].feature_importances_\n    feats= backend[\"feats\"]; flab = backend[\"feat_labels\"]\n    fig_fi, ax_fi = plt.subplots(figsize=(7,4))\n    fig_fi.patch.set_facecolor(\"#0F1923\"); ax_fi.set_facecolor(\"#0F1923\")\n    idx = np.argsort(fi)\n    ax_fi.barh([flab.get(feats[i],feats[i]) for i in idx], fi[idx],\n                color=BLUE, edgecolor=\"none\")\n    ax_fi.set_xlabel(\"Importance\", color=\"white\")\n    ax_fi.set_title(\"GBM Feature Importance\", color=\"white\", fontweight=\"bold\")\n    ax_fi.tick_params(colors=\"white\")\n    ax_fi.spines[\"bottom\"].set_color(\"#444\"); ax_fi.spines[\"left\"].set_color(\"#444\")\n    for sp in [\"top\",\"right\"]: ax_fi.spines[sp].set_visible(False)\n    plt.tight_layout()\n    st.pyplot(fig_fi, use_container_width=True); plt.close(fig_fi)\n\n    st.markdown(\"---\")\n    st.subheader(\"XAI Method Reference Guide\")\n    method_info = [\n        (GREEN,  \"SHAP\",           \"NB2\", \"Shapley values\",\n         \"Theoretically grounded (4 axioms). Deterministic. Gold standard for tabular GBM.\"),\n        (BLUE,   \"LIME\",           \"NB3\", \"Local linear surrogate\",\n         \"Fast and model-agnostic. Can be unstable across runs. Good for quick prototyping.\"),\n        (TEAL,   \"MAPLE\",          \"NB3\", \"RF leaf co-membership\",\n         \"More stable than LIME. Uses model structure for neighbourhood. Better for audits.\"),\n        (ORANGE, \"Counterfactual\", \"NB4\", \"Nearest opposite patient\",\n         \"Most actionable. Answers: what would need to change? Best for patient communication.\"),\n        (BROWN,  \"Surrogate Tree\", \"NB4\", \"Global IF-THEN rules\",\n         \"Human-readable globally. Can be printed on a clinical decision card.\"),\n        (PURPLE, \"GEMEX\",          \"NB22\",\"Fisher-Rao geodesic\",\n         \"Information-geometric. Measures structural deviation from reference group.\"),\n    ]\n    cols = st.columns(3)\n    for i, (col, name, nb_ref, mtype, desc) in enumerate(method_info):\n        with cols[i % 3]:\n            st.markdown(\n                f'<div style=\"background:#131326;padding:12px;border-radius:8px;'\n                f'border:2px solid {col};margin-bottom:10px\">'\n                f'<b style=\"color:{col}\">{name}</b>'\n                f'<span style=\"color:#555;font-size:.72em\"> . {nb_ref} . {mtype}</span>'\n                f'<p style=\"color:#9999BB;font-size:.80em;margin:6px 0 0\">{desc}</p>'\n                f'</div>', unsafe_allow_html=True)\n\n    if not GEMEX_OK:\n        st.warning(\"GEMEX not installed. Install with: pip install gemex\")\n"

with open('app_nb23.py', 'w') as f:
    f.write(app_code)

import os
size = os.path.getsize('app_nb23.py')
print(f'app_nb23.py written ({size:,} bytes / {size//1024} KB)')
print()
print('Launch with:')
print('  python -m streamlit run app_nb23.py')
print()
print('App contains 6 XAI methods:')
print('  1. SHAP            -- Shapley values')
print('  2. LIME            -- Local linear surrogate')
print('  3. MAPLE           -- RF co-membership surrogate')
print('  4. Counterfactual  -- Nearest opposite patient')
print('  5. Surrogate Tree  -- Global IF-THEN rules')
print('  6. GEMEX           -- Geodesic information-geometric')

app_nb23.py written (38,304 bytes / 37 KB)

Launch with:
  python -m streamlit run app_nb23.py

App contains 6 XAI methods:
  1. SHAP            -- Shapley values
  2. LIME            -- Local linear surrogate
  3. MAPLE           -- RF co-membership surrogate
  4. Counterfactual  -- Nearest opposite patient
  5. Surrogate Tree  -- Global IF-THEN rules
  6. GEMEX           -- Geodesic information-geometric


---
## Section 4 -- Deployment

### Local
```bash
pip install streamlit shap lime gemex scikit-learn pandas numpy matplotlib
streamlit run app_nb23.py
# Opens at http://localhost:8501
```

### Google Colab
```python
!pip install streamlit pyngrok shap lime scikit-learn -q
from pyngrok import ngrok
ngrok.set_auth_token('YOUR_FREE_TOKEN')   # free at ngrok.com
!streamlit run app_nb23.py &
import time; time.sleep(4)
print(ngrok.connect(8501))               # prints public URL
```

### Streamlit Community Cloud (free, zero infrastructure)
1. Push `app_nb23.py` + `requirements.txt` to a public GitHub repo
2. Go to [share.streamlit.io](https://share.streamlit.io) -> New app -> connect repo
3. Click **Deploy** -- live public URL in ~2 minutes

`requirements.txt`:
```
streamlit
shap
lime
gemex
scikit-learn
pandas
numpy
matplotlib
```

---
## Summary

| Section | Content | Key deliverable |
|---------|---------|------------------|
| S1 | Five Streamlit core concepts | `cache_resource` . `sidebar` . `tabs` . `session_state` . `download_button` |
| S2 | Backend verification | Cleveland AUC + Pima AUC + GEMEX import check |
| S3 | `app_nb23.py` | 4-tab dashboard, 6 XAI methods, CSV upload, comparison tab |
| S4 | Deployment | Local (python -m streamlit) . Colab/ngrok . Community Cloud |

**6 XAI Methods -- course coverage in one dashboard:**

| Method | Type | Best for |
|--------|------|----------|
| SHAP | Shapley attribution | Gold standard, deterministic |
| LIME | Local linear surrogate | Quick prototyping |
| MAPLE | RF co-membership | Stable local explanations |
| Counterfactual | Nearest opposite | Patient communication |
| Surrogate Tree | IF-THEN rules | Clinical decision cards |
| GEMEX |  Geodesic deviation | Information-geometric novelty |